# Laya × Memory Fusion V2 (Bidirectional)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_MemoryFusion_Colab.ipynb)

This notebook keeps **'convaiinnovations/laya' unchanged as the teacher** and ports the successful TinyCeNN MemoryFusion training recipe to Laya's bidirectional ModernBERT encoder.

### What changed from the first Laya MemoryFusion run
- Reuses the repo's successful **Cellular + Hedgehog + GDN2 MemoryFusion core**.
- Runs that core forward and reverse with shared weights for bidirectional Laya.
- Local-heavy fusion initialization '[2, -1, -1]' from the successful MemoryFusion implementation.
- 'memory_rank=64', 'feature_dim=32'.
- Laya 'Wqkv' stays frozen; 'Wo' is calibrated with a small learning rate.
- Teacher→student hidden-state curriculum ('alpha 0.9 → 0').
- Functional loss + Laya decision-logit KL + action KL.
- Up to four resumable 300-step rounds per layer; failed rounds **keep their weights**.
- Acceptance is measured on **real current-student hidden states**.
- Gate and final evaluation sets are disjoint and workflow-stratified.
- Phase 1 targets only the strongest 'full_attention' layers: **12, then 15**. Sliding attention is not silently approximated.


## 1. Setup
A T4/L4/A100 runtime is recommended. The setup pulls the latest TinyCeNN-LM and Laya source, then performs a syntax preflight before training.


In [ ]:
import os, sys, subprocess, pathlib, importlib, compileall
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REPO = WORK / "TinyCeNN-LM"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "-q"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/NandhaKishorM/laya.git",
    "datasets", "pandas", "pyarrow", "safetensors"
])

SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
assert compileall.compile_dir(str(LAB_SRC), quiet=1), "Python syntax preflight failed in tinycenn_lm/laya_lab"

import torch, json, pandas as pd
from tinycenn_lm.laya_lab.memory_fusion_v2 import (
    LayaMemoryFusionV2Config,
    run_memory_fusion_v2,
)

print("repo:", REPO)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Configuration

The default run deliberately starts with full-attention layer **12**, then layer **15** if layer 12 passes or exhausts its rounds. The strict gates are kept; the trainer improves the model rather than loosening acceptance.


In [ ]:
MODEL_ID = "convaiinnovations/laya"  # keep the original Laya teacher
CANDIDATE_LAYERS = (12, 15)

cfg = LayaMemoryFusionV2Config(
    model_id=MODEL_ID,
    seed=2026,
    output_dir="/content/laya_tinycenn",

    candidate_layers=CANDIDATE_LAYERS,
    feature_dim=32,
    memory_rank=64,
    dilations=(1, 2, 4, 8, 16, 32, 64),

    train_cases=400,
    train_max_len=512,
    batch_size=2,
    steps_per_round=300,
    max_rounds=4,
    check_every=25,
    min_steps_before_check=50,

    core_learning_rate=2e-4,
    output_learning_rate=2e-5,
    teacher_alpha_start=0.90,
    resume_alpha_start=0.25,
    teacher_alpha_end=0.0,

    max_local_nmse=0.20,
    min_local_cosine=0.90,
    min_teacher_agreement=0.95,
    max_mean_kl=0.05,
    max_accuracy_drop=0.02,

    gate_cases=80,
    final_cases=160,
)
print(cfg)


## 3. Train / resume-style sequential acceptance

Each candidate is trained for up to 300 steps per round. If it has not passed, its current weights are saved and training continues from those weights in the next round. A layer is accepted only when both local functional fidelity and Laya decision-level gates pass.


In [ ]:
teacher, student, report = run_memory_fusion_v2(cfg)


## 4. Result summary
If no layer passes, the final student is restored to original Laya attention and the report explicitly says 'failed_no_accepted_layers'. Identical teacher/student metrics are therefore never presented as a successful conversion.


In [ ]:
summary = pd.DataFrame([
    {
        "model": "Laya teacher",
        **{k: report["teacher_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
    },
    {
        "model": report["architecture"],
        **{k: report["student_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
        "teacher_agreement": report["student_final"].get("teacher_agreement"),
        "teacher_KL": report["student_final"].get("mean_teacher_kl"),
    },
])
display(summary)

print("Status:", report["status"])
print("Accepted attention layers:", report["accepted_layers"])
print("Replacement trainable parameters:", f'{report["replacement_trainable_parameters"]:,}')
print("Gate/final disjoint:", report["gate_final_disjoint"])
print("Latency:", json.dumps(report["latency"], indent=2))


## 5. Round-by-round diagnostics

This table is the key diagnostic. A layer can continue across multiple rounds instead of being discarded after one short fit.


In [ ]:
rows = []
for h in report["history"]:
    gate = h.get("gate") or {}
    local = h.get("local") or {}
    rows.append({
        "layer": h["layer"],
        "round": h["round"],
        "accepted": h["accepted"],
        "steps": h["steps"],
        "nmse": local.get("nmse"),
        "cosine": local.get("cosine"),
        "teacher_agreement": gate.get("teacher_agreement"),
        "mean_teacher_kl": gate.get("mean_teacher_kl"),
        "accuracy": gate.get("accuracy"),
        "accuracy_drop": h.get("accuracy_drop"),
    })
rounds = pd.DataFrame(rows)
display(rounds)


## 6. Laya Router smoke test

The adapted Agent keeps Laya's public API. The example below attaches the already-loaded student without loading a second model.


In [ ]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.",
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
    "refund_requested": {
        "type": "noul",
        "instructions": "Does the user explicitly request a refund?",
    },
}

router = Router(preload=False)
router.attach("english", student)
res = router.predict(state, questions, model="english")

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


## 7. Saved outputs

Accepted adapters are written under '/content/laya_tinycenn/memory_fusion_v2/'. Every training round also saves a layer checkpoint, so a difficult layer is not lost after one run.


In [ ]:
from pathlib import Path
out = Path(cfg.output_dir) / "memory_fusion_v2"
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print("Round checkpoints:")
for p in sorted(out.glob("layer_*_round_*.pt")):
    print(" -", p.name)
print("\nReport preview:")
print((out / "report.json").read_text()[:5000])
